# Notebook 4 — Lead Optimization Agent · LangChain + OpenRouter

Same chemistry task as Notebook 2, rebuilt with the LangChain stack.

| | Notebook 2 | **This notebook** |
|---|---|---|
| LLM client | `anthropic` SDK | `langchain-openai` → OpenRouter |
| Tool format | Raw JSON schema | `@tool` decorator |
| Agent loop | Manual `while` loop | `AgentExecutor` |
| Model routing | Anthropic direct | Any OpenRouter model |

**What OpenRouter adds:** swap between Claude, GPT-4o, Gemini, Llama, etc. by changing one string.

> All chemistry scoring still runs locally via RDKit — no extra API needed for that part.
>
> Tool names (`validate_smiles`, `analyze_molecule`, `compare_candidates`) match `hermes_tools/lead_opt.py` exactly.

---
## 0. Install Dependencies

In [1]:
# Run once — comment out after first install
# !pip install langchain langchain-openai langchain-core python-dotenv

---
## 1. Imports

In [1]:
import sys, os, json
from dotenv import load_dotenv

# Load OPENROUTER_API_KEY from ../.env
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from agent_utils import (
    call_admet_api,
    is_valid_smiles,
    compare_candidates as _compare_candidates,
)

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

print('All imports OK')

All imports OK


---
## 2. Connect to OpenRouter

OpenRouter exposes an **OpenAI-compatible API**, so `ChatOpenAI` works out of the box.
Just point `openai_api_base` at OpenRouter and pass your key.

Browse available models at **https://openrouter.ai/models**.

In [3]:
MODEL = "deepseek/deepseek-v4-flash"   # ← change to any OpenRouter model
# Other options (confirmed available on OpenRouter):
#   "anthropic/claude-haiku-4.5"             (fast + cheap)
#   "anthropic/claude-opus-4.6"              (most capable)
#   "deepseek/deepseek-v4-flash"             (fast, cost-effective)
#   "openai/gpt-4o"
#   "google/gemini-2.0-flash-001"
#   "meta-llama/llama-3.3-70b-instruct"      (open-weights)

llm = ChatOpenAI(
    model=MODEL,
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
    max_tokens=4096,
)

# Quick smoke-test
resp = llm.invoke("Reply with exactly: OK")
print("LLM test:", resp.content)

LLM test: OK


---
## 3. Define Tools

LangChain's `@tool` decorator turns a Python function into a tool.
- The **function name** becomes the tool name the LLM calls
- The **docstring** becomes the description
- **Type hints** define the input schema automatically — no manual JSON schema needed

Names match `hermes_tools/lead_opt.py` exactly: `validate_smiles`, `analyze_molecule`, `compare_candidates`.

In [4]:
@tool
def validate_smiles(smiles: str) -> str:
    """Validate a SMILES string and return its canonical form. Always call this before analyze_molecule."""
    return json.dumps(is_valid_smiles(smiles))


@tool
def analyze_molecule(smiles: str) -> str:
    """
    Compute a full ADMET profile for a molecule (local RDKit, no API).
    Returns MW, cLogP, TPSA, QED, BBB penetration, CNS MPO, solubility,
    Lipinski rules, PAINS alerts, synthetic accessibility, and decision guidance.
    """
    result = call_admet_api(smiles)   # already returns the flat scored dict
    return json.dumps(result)


@tool
def compare_candidates(smiles_list: list, labels: list = None) -> str:
    """Compare multiple drug candidates side by side. Returns ADMET scores for all molecules in one call."""
    return json.dumps(_compare_candidates(smiles_list, labels))


TOOLS = [validate_smiles, analyze_molecule, compare_candidates]

print(f"{len(TOOLS)} tools registered:")
for t in TOOLS:
    print(f"  • {t.name}")

3 tools registered:
  • validate_smiles
  • analyze_molecule
  • compare_candidates


---
## 4. Build the Agent

Three LangChain pieces:

1. **Prompt** — `ChatPromptTemplate` with a `{agent_scratchpad}` slot (LangChain inserts tool call history here)
2. **Agent** — `create_tool_calling_agent(llm, tools, prompt)` wires them together
3. **Executor** — `AgentExecutor` runs the loop: call LLM → execute tool → feed result back → repeat

In [5]:
SYSTEM_PROMPT = """
You are an expert medicinal chemistry AI specializing in lead optimization.
Your job: iteratively improve a drug candidate's ADMET profile toward a user-specified goal.

## Workflow (follow strictly each round)
1. Identify the biggest gap between current scores and the goal
2. Reason which structural change addresses that gap
3. Propose a modified SMILES with explicit chemical justification
4. validate_smiles → if invalid, try a simpler change
5. analyze_molecule → get new ADMET scores
6. compare_candidates every 2 rounds to track overall progress
7. Stop when ALL goal criteria are met OR no improvement in 2 consecutive rounds

## Key structure-property rules
| Goal                  | Strategy |
|-----------------------|---------|
| Improve BBB / CNS MPO | Reduce HBD (replace -OH/-NH2 with -F/-OMe), reduce MW (<450), reduce PSA (<90 Å²) |
| Improve solubility    | Add -OH, -NH2, ionisable groups; reduce aromaticity |
| Improve QED           | Optimise MW (200-500), LogP (0-5), reduce rotatable bonds |
| Remove alerts         | Avoid quinones, catechols, Michael acceptors, anilines |
| Bioisosteres          | -COOH ↔ tetrazole, -OH ↔ -F, phenyl ↔ pyridine |

Make ONE structural change per round for clear SAR understanding.
End with: best candidate SMILES + summary table + chemical reasoning.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),  # LangChain inserts tool history here
])

print("Prompt template ready")

Prompt template ready


In [6]:
agent = create_tool_calling_agent(llm, TOOLS, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,                   # prints every tool call + result
    max_iterations=15,              # safety brake
    return_intermediate_steps=True, # capture tool call history for results table
)

print("AgentExecutor ready")
print(f"  Model    : {MODEL}")
print(f"  Max iters: {agent_executor.max_iterations}")

AgentExecutor ready
  Model    : deepseek/deepseek-v4-flash
  Max iters: 15


---
## 5. Run!

**Scenario:** Optimize **Atenolol** for CNS penetration.

| Property | Atenolol (start) | Propranolol (reference) | Target |
|---|---|---|---|
| cLogP | 0.45 | 2.58 | > 1.5 |
| TPSA (Å²) | 84.6 | 41.5 | < 70 |
| HBD | 3 | 2 | ≤ 2 |
| BBB probability | 0.60 | 0.90 | > 0.80 |
| CNS MPO | 4.55 | 5.45 | > 5.0 |
| QED | 0.638 | 0.838 | > 0.75 |

The agent must reduce polarity (para-amide is the main liability) while preserving the aryloxypropanolamine pharmacophore.

In [7]:
STARTING_SMILES = "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1"  # Atenolol

GOAL = """
Starting molecule: Atenolol  SMILES: CC(C)NCC(O)COc1ccc(CC(N)=O)cc1

Target profile (reference: Propranolol):
  - BBB probability  > 0.80  (currently 0.60)
  - CNS MPO score    > 5.0   (currently 4.55)
  - QED              > 0.75  (currently 0.638)
  - Rotatable bonds  ≤ 6     (currently 8)
  - Zero structural alerts
  - MW < 400 Da

Chemistry hint: the para-amide (-CC(N)=O) is the biggest polarity liability.
Replacing or removing it should lower HBD and TPSA significantly.
Preserve the aryloxypropanolamine core (required for beta-blocker activity).
"""

print("Starting optimization...\n")
result = agent_executor.invoke({"input": GOAL})

Starting optimization...



> Entering new AgentExecutor chain...

Invoking: `validate_smiles` with `{'smiles': 'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1'}`


{"valid": true, "canonical_smiles": "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1", "num_heavy_atoms": 19}
Invoking: `analyze_molecule` with `{'smiles': 'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1'}`


{"canonical_smiles": "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1", "molecular_weight": 266.2, "clogp": 0.45, "tpsa": 84.6, "hbd": 3, "hba": 4, "rotatable_bonds": 8, "qed_score": 0.638, "qed_class": "Moderate", "lipinski_summary": "Pass", "fail_fast_score": 0.0, "decision": "Optimize", "decision_rationale": "Some properties need improvement: QED 0.64, CNS MPO 2.3, BBB prob 0.18. Targeted structural changes can address the gaps.", "solubility_class": "Soluble", "log_s": -1.48, "gi_absorption": "High", "bbb_penetrates": false, "bbb_probability": 0.177, "cns_mpo_score": 2.27, "cns_class": "CNS-", "alerts": [], "num_alerts": 0, "cyp3a4_substrate": false, "cyp_inhibitor_risk": false, 

---
## 6. Results

In [8]:
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["output"])

FINAL ANSWER
Agent stopped due to max iterations.


In [9]:
import pandas as pd

# Extract all analyzed molecules from intermediate steps
candidates = []
for action, observation in result["intermediate_steps"]:
    if action.tool == "analyze_molecule":
        try:
            data = json.loads(observation)
            if "error" not in data:
                data["input_smiles"] = action.tool_input["smiles"]
                candidates.append(data)
        except Exception:
            pass

if candidates:
    df = pd.DataFrame(candidates)
    cols = [
        "input_smiles", "molecular_weight", "clogp",
        "qed_score", "bbb_probability", "cns_mpo_score",
        "num_alerts", "decision"
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols])

    start = df.iloc[0]
    best  = df.loc[df["cns_mpo_score"].idxmax()]
    print(f"\nCNS MPO : {start['cns_mpo_score']:.2f} → {best['cns_mpo_score']:.2f}")
    print(f"BBB prob: {start['bbb_probability']:.3f} → {best['bbb_probability']:.3f}")
    print(f"QED     : {start['qed_score']:.3f} → {best['qed_score']:.3f}")
else:
    print("No candidates extracted — check intermediate_steps above.")

,input_smiles,molecular_weight,clogp,qed_score,bbb_probability,cns_mpo_score,num_alerts,decision
0,CC(C)NCC(O)COc1ccc(CC(N)=O)cc1,266.2,0.45,0.638,0.177,2.27,0,Optimize
1,CC(C)NCC(O)COc1ccc(CC)cc1,237.2,1.99,0.762,0.634,3.96,0,Optimize
2,CC(C)NCC(O)COc1ccc(C)cc1,223.2,1.73,0.771,0.632,3.83,0,Optimize
3,CC(C)N(C)CC(O)COc1ccc(C)cc1,237.2,2.07,0.822,0.730,4.37,0,Progress
4,CC(C)N(C)CC(O)COc1ccc(CC)cc1,251.2,2.33,0.807,0.732,4.50,0,Progress
5,CC(C)N(C)CC(OC)COc1ccc(C)cc1,251.2,2.73,0.744,0.830,4.87,0,Progress



CNS MPO : 2.27 → 4.87
BBB prob: 0.177 → 0.830
QED     : 0.638 → 0.744


In [12]:
# Save for visualization notebook
output_path = "../saved_runs/latest_run.json"
with open(output_path, "w") as f:
    json.dump(candidates, f, indent=2, default=str)
print(f"Saved {len(candidates)} candidates → {output_path}")

Saved 6 candidates → ../saved_runs/latest_run.json


---
## How it works

```
agent_executor.invoke({"input": GOAL})
        │
        ▼
  ChatOpenAI (OpenRouter)  ←──  system prompt + {input}
        │
        ▼
  Tool call requested?  ──Yes──►  execute tool (RDKit, local)
        │                                │
        │◄──────  tool_result ◄──────────┘
        │
  LLM reasons again with result in context
        │
  stop_reason == end_turn?  ──Yes──►  return final answer
        │
       No → repeat (up to max_iterations)
```

**Key LangChain concepts used:**

| Concept | What it does |
|---|---|
| `@tool` | Wraps a Python function into a LangChain tool (schema auto-generated from type hints + docstring) |
| `ChatPromptTemplate` | Structures system prompt + user input + scratchpad slot |
| `create_tool_calling_agent` | Connects LLM + tools + prompt into a runnable agent |
| `AgentExecutor` | Handles the loop: LLM call → tool dispatch → feed result back → repeat |
| `return_intermediate_steps=True` | Exposes every tool call + result for inspection |

**Switching models:** change `MODEL` in cell 2 — no other code changes needed.

➡️ **Next:** `03_visualization.ipynb` — plot the optimization trajectory